### RAG pipeline- Data Ingestion to Vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: DBMS Unit 5 Notes.pdf
  ✓ Loaded 23 pages

Processing: OS-UNIT 1.pdf
  ✓ Loaded 27 pages

Total documents loaded: 50


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2020-05-28T20:40:46+05:30', 'moddate': '2024-07-28T17:38:33+05:30', 'author': 'Ravindar', 'source': '..\\data\\pdf\\DBMS Unit 5 Notes.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'DBMS Unit 5 Notes.pdf', 'file_type': 'pdf'}, page_content='Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 1 \n \nUNIT – V \nData on External Storage, File Organization and Indexing, Cluster Indexes, Primary and Secondary Indexes, Index data \nStructures, Hash Based Indexing, Tree base Indexing, Comparison of File Organizations, Indexes and Performance Tuning, \nIntuitions for tree Indexes, Indexed Sequential Access Methods (ISAM), B+ Trees: A Dynamic Index Structure. \n1. DATA ON EXTERNAL STORAGE \nPrimary memory has limited storage capacity and is volatile. To overcome this limitation, \nsecondary memory is also termed as external storage devic

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 50 documents into 118 chunks

Example chunk:
Content: Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 1 
 
UNIT – V 
Data on External Storage, File Organization and Indexing, Cluster Indexes, Primary and Secondary Indexes, Index data 
...
Metadata: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2020-05-28T20:40:46+05:30', 'moddate': '2024-07-28T17:38:33+05:30', 'author': 'Ravindar', 'source': '..\\data\\pdf\\DBMS Unit 5 Notes.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'DBMS Unit 5 Notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2020-05-28T20:40:46+05:30', 'moddate': '2024-07-28T17:38:33+05:30', 'author': 'Ravindar', 'source': '..\\data\\pdf\\DBMS Unit 5 Notes.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'DBMS Unit 5 Notes.pdf', 'file_type': 'pdf'}, page_content='Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 1 \n \nUNIT – V \nData on External Storage, File Organization and Indexing, Cluster Indexes, Primary and Secondary Indexes, Index data \nStructures, Hash Based Indexing, Tree base Indexing, Comparison of File Organizations, Indexes and Performance Tuning, \nIntuitions for tree Indexes, Indexed Sequential Access Methods (ISAM), B+ Trees: A Dynamic Index Structure. \n1. DATA ON EXTERNAL STORAGE \nPrimary memory has limited storage capacity and is volatile. To overcome this limitation, \nsecondary memory is also termed as external storage devic

### Embedding and VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid  #every record will have unique id which is inserted into vector DB 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\saray\my-rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    ## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\saray\my-rag-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saray\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3478.34it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\saray\AppData\Local\Temp\ipykernel_1980\3043582926.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(
                f"Vector store initialized. Collection: {self.collection_name}"
            )
            print(
                f"Existing documents in collection: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 1 \n \nUNIT – V \nData on External Storage, File Organization and Indexing, Cluster Indexes, Primary and Secondary Indexes, Index data \nStructures, Hash Based Indexing, Tree base Indexing, Comparison of File Organizations, Indexes and Performance Tuning, \nIntuitions for tree Indexes, Indexed Sequential Access Methods (ISAM), B+ Trees: A Dynamic Index Structure. \n1. DATA ON EXTERNAL STORAGE \nPrimary memory has limited storage capacity and is volatile. To overcome this limitation, \nsecondary memory is also termed as external storage devices are used.  External storage devices \nsuch as disks and tapes are used to store data permanently.  \nThe Secondary storage devices can be fixed or removable. Fixed Storage device is an \ninternal storage device like hard disk that is fixed inside the computer. Storage devices that are \nportable and can be taken outside the computer are termed as removable storage devices such a

In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

##store int he vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 118 texts...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]

Generated embeddings with shape: (118, 384)
Adding 118 documents to vector store...
Successfully added 118 documents to vector store
Total documents in collection: 118


### Retriever Pipeline from VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents (after filtering)"
                )
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [14]:
rag_retriever

In [15]:
# Question about system calls
rag_retriever.retrieve("What are the system calls fork, exec, and wait?")

Retrieving documents for query: 'What are the system calls fork, exec, and wait?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5e5d1cd5_93',
  'content': 'open() before reading. \nwait() \nIn some systems, a process may have to wait for another process to complete its \nexecution before proceeding. When a parent process makes a child process, the parent \nprocess execution is suspended until the child process is finished. The  wait() system call is \nused to suspend the  parent process. Once the child  process has completed its execution, \ncontrol is returned to the parent process. \nwrite() \nIt is used to write data from a user buffer to a device like a file. This system call is one \nway for a program to generate data. It takes three arguments in general: \n\uf0d8 A file descriptor. \n\uf0d8 A pointer to the buffer in which data is saved. \n\uf0d8 The number of bytes to be written from the buffer. \nfork() \nProcesses generate clones of themselves using the fork() system call. It is one of the \nmost common wa ys to create processes in operating systems. When a parent process',
  'metadata': {

In [16]:
# Question about B+ Trees
rag_retriever.retrieve("How does search and insertion work in a B+ tree?")

Retrieving documents for query: 'How does search and insertion work in a B+ tree?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_13329ace_40',
  'content': 'Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 17 \n \nAfter inserting 24, 33, 36, and 39 in the above tree, it looks like \n \nDeletion: From the above figure, after deleting 42, 71, 24 and 36 \n \n \n9. B+ TREE \nB+ Tree is an extension of Binary Tree which allows efficient insertion, deletion and search \noperations. It is used to implement indexing in DBMS. In B+ tree, data can be stored only o n the \nleaf nodes while internal nodes can store the search key values.  \n1. B+ tree of an order m can store max m-1 values at each node. \n2. Each node can have a maximum of m children and at least m/2 children (except root). \n3. The values in each node are in sorted order. \n4. All the nodes must contain at least half full except the root node. \n5. Only leaf nodes contain values and non-leaf nodes contain search keys. \n    31   \n  68  23     59  42   \n 20  10  27  23  35  31  46    61  59    68 \n  33 \n   39 \n    31   \n

### Integration VectorDB Context Pipeline with LLM output

In [22]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# 1. Initialize Groq with a highly stable Llama 3 model
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant",  # Updated to official stable model name
    temperature=0.1,
    max_tokens=1024,
)


# 2. Updated RAG Function
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)

    # Format retrieved context
    context = (
        "\n\n".join([doc["content"] for doc in results]) if results else ""
    )

    if not context:
        return "No relevant context found to answer the question."

    # Standard clean prompt string
    prompt = (
        f"Use the following context to answer the question concisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )

    # Pass the prompt string directly to llm.invoke()
    response = llm.invoke(prompt)
    return response.content

In [23]:
# Option 1: File organization
answer = rag_simple(
    "What are the different types of file organization?", rag_retriever, llm
)
print(answer)

Retrieving documents for query: 'What are the different types of file organization?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The different types of file organization are:

1. Heap File Organization
2. Sequential File Organization
3. Hash File Organization


### Enhanced RAG Pipeline features


In [24]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(
    query, retriever, llm, top_k=5, min_score=0.2, return_context=False
):
    """RAG pipeline with extra features:

    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(
        query, top_k=top_k, score_threshold=min_score
    )
    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": "",
        }

    # Prepare context and sources
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get(
                "source_file", doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "...",
        }
        for doc in results
    ]
    confidence = max([doc["similarity_score"] for doc in results])

    # Generate answer using clean string formatting
    prompt = (
        f"Use the following context to answer the question concisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )

    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence,
    }

    if return_context:
        output["context"] = context

    return output
# Example usage:
result = rag_advanced(
    "What are the different types of file organization?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)

print("Answer:", result["answer"])
print("\nSources:", result["sources"])
print("\nConfidence:", result["confidence"])
print("\nContext Preview:", result["context"][:300])

Retrieving documents for query: 'What are the different types of file organization?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: There are three types of file organization:

1. Heap File Organization
2. Sequential File Organization
3. Hash File Organization

Sources: [{'source': 'DBMS Unit 5 Notes.pdf', 'page': 0, 'score': 0.19347983598709106, 'preview': 'internal storage device like hard disk that is fixed inside the computer. Storage devices that are \nportable and can be taken outside the computer are termed as removable storage devices such as \nCD, DVD, external hard disk, etc. \nMagnetic/optical Disk: It supports random and sequential access. It t...'}, {'source': 'DBMS Unit 5 Notes.pdf', 'page': 1, 'score': 0.16750609874725342, 'preview': 'Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 2 \n \n \nHeap File Organization : When a file is created using Heap File Organization mechanism, the  \nrecords are stored in the file in the order in which they are inserted. So the new records are \ninserted at the end of the file. In ...'}]

Confidence: 0.19347983598709106

Context Preview: i

In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self, 
        question: str, 
        top_k: int = 5, 
        min_score: float = 0.2, 
        stream: bool = False, 
        summarize: bool = False
    ) -> Dict[str, Any]:
        
        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score
        )
        
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            # Standard prompt string (fixes the invoke error)
            prompt = (
                f"Use the following context to answer the question concisely.\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer:"
            )

            # Streaming answer simulation
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})" 
            for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations) 
            if citations else answer
        )

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }
# Initialize the pipeline
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

# Execute query (tailored to DBMS / OS notes)
result = adv_rag.query(
    "What are the different types of file organization?", 
    top_k=3, 
    min_score=0.1, 
    stream=True, 
    summarize=True
)

# Print outputs
print("\nFinal Answer:\n", result['answer'])
print("\nSummary:\n", result['summary'])
print("\nHistory (Latest Entry):\n", result['history'][-1])

Retrieving documents for query: 'What are the different types of file organization?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 92.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.

Context:
internal storage device like hard disk that is fixed inside the computer. Storage devices 

that are 
portable and can be taken outside the computer are termed as removable storage devices such as 
CD, DVD, external hard disk, etc. 
Magnetic/optical Disk: It supports random and sequential access. It takes less access time. 
Magnetic Tapes: It supports only sequential access. It takes more access time. 
In DBMS, processing a query and getting output need accessing random pages.  So, disks 
are preferable than magnetic tapes. 
2. FILE ORGANIZATION 
The database is stored as a collection of files. Each file contains a set of records. Each record 
is a collection of fields. For example, a student table (or file) contains many records and each 
record belongs to one student with fields (attributes) such as Name, Date of birth, class, 
department, address, etc.  
File organization defines how file records are mapped onto disk blocks.

Prepared by: Ravindar Mogili, Associate Professor, JITS-KNR  Page: 2 
 
 
Heap File Organization : When a file is created using Heap File Organizatio